# Sicilian NMT — back-translation (NLLB-1.3B)

Augment **scn→en** with back-translation (Sennrich 2016). Steps:
1. Load NLLB-1.3B + our **bidirectional LoRA adapter** (`nllb-lora-bidir-1.3B`), merged in.
2. Back-translate in-domain **monolingual English** (`mono_en.txt`, harvested from the
   unaligned Arba Sicula text) **en→scn** → synthetic Sicilian.
3. Fine-tune **scn→en** on real (27k) + synthetic pairs, evaluate on the frozen test.

The English targets are real/clean; only the Sicilian source is synthetic — this teaches
the decoder more fluent English and regularizes the forward model. Best so far (scn→en,
frozen test): bidirectional LoRA **31.33 BLEU**; the paper's augmented number is 36.8.

In [ ]:
!pip -q install transformers datasets peft sentencepiece sacrebleu accelerate
!pip -q uninstall -y torchao   # Colab's old torchao breaks peft's LoRA import

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/SicilianNMT-colab'
except Exception:
    OUT = 'sicilian-nllb-out'
os.makedirs(OUT, exist_ok=True)
ADAPTER = f'{OUT}/nllb-lora-bidir-1.3B'   # produced by the bidirectional notebook
print('using', OUT, '| adapter exists:', os.path.exists(ADAPTER))

In [ ]:
def read(p): return open(p, encoding='utf-8').read().splitlines()
DATA = f'{OUT}/data'
base = DATA if os.path.exists(f'{DATA}/train.scn') else '.'
print('reading data from', base)
splits = {s: {l: read(f'{base}/{s}.{l}') for l in ('scn', 'en')} for s in ('train', 'valid', 'test')}
# monolingual English to back-translate (harvest_mono_en.py output).
# Looked up on Drive AND in /content, so you can just drag mono_en.txt into the
# Colab Files panel if Insync hasn't pushed it to Drive yet.
_cands = [f'{base}/mono_en.txt', f'{DATA}/mono_en.txt', 'mono_en.txt', '/content/mono_en.txt']
MONO = next((p for p in _cands if os.path.exists(p)), None)
mono_en = read(MONO) if MONO else []
if not mono_en:
    print('WARNING: mono_en.txt not found. Drag it into the left Files panel (into /content), '
          'then re-run this cell. Looked in:', _cands)
print({s: len(d['scn']) for s, d in splits.items()}, '| mono_en:', len(mono_en), 'from', MONO)

In [ ]:
import os, gc, json, torch
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel
from sacrebleu.metrics import BLEU, CHRF

for _v in ('model', 'ft', 'base_model'):
    if _v in globals():
        del globals()[_v]
gc.collect();
if torch.cuda.is_available(): torch.cuda.empty_cache()

MODEL = 'facebook/nllb-200-1.3B'
LANG = {'scn': 'scn_Latn', 'en': 'eng_Latn'}
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if device == 'cuda' else torch.float32
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL, torch_dtype=dtype, low_cpu_mem_usage=True)
if os.path.exists(ADAPTER):
    model = PeftModel.from_pretrained(model, ADAPTER)
    model = model.merge_and_unload()   # bake the bidirectional adapter into the weights
    print('merged bidirectional adapter')
model = model.to(device).eval()
results = {}

def translate(texts, src, tgt, bs=8, max_len=160, beams=5):
    tok.src_lang = LANG[src]
    tgt_id = tok.convert_tokens_to_ids(LANG[tgt])
    out = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i+bs], return_tensors='pt', padding=True, truncation=True, max_length=max_len).to(device)
        with torch.no_grad():
            gen = model.generate(**enc, forced_bos_token_id=tgt_id, max_length=max_len, num_beams=beams)
        out += tok.batch_decode(gen, skip_special_tokens=True)
        print(f'  {min(i+bs, len(texts))}/{len(texts)}', end='\r')
    print(); torch.cuda.empty_cache()
    return out

def report(hyp, ref, tag):
    b = BLEU().corpus_score(hyp, [ref]).score
    c = CHRF().corpus_score(hyp, [ref]).score
    results[tag] = {'bleu': round(b, 2), 'chrf': round(c, 2)}
    open(f"{OUT}/hyp_{tag.replace(' ', '_').replace('->', '2')}.txt", 'w', encoding='utf-8').write('\n'.join(hyp) + '\n')
    json.dump(results, open(f'{OUT}/results_bt.json', 'w'), indent=2, ensure_ascii=False)
    print(f'{tag}:  BLEU={b:.2f}  chrF={c:.2f}')
print('ready on', device, '|', dtype)

## Back-translate monolingual English → synthetic Sicilian

Cap the pool with `MAX_MONO` (beam search over many sentences is slow). Synthetic pairs
are filtered by length ratio and deduped before training.

In [ ]:
MAX_MONO = 15000
pool = mono_en[:MAX_MONO]
print(f'back-translating {len(pool)} English sentences en->scn ...')
synth_scn = translate(pool, 'en', 'scn', bs=16, beams=4) if pool else []

# build synthetic (scn, en) pairs: real English target, synthetic Sicilian source
syn_pairs, seen = [], set(s for s in splits['train']['scn'])
for sc, en in zip(synth_scn, pool):
    sc, en = sc.strip(), en.strip()
    if not sc or not en or sc in seen:
        continue
    r = len(sc.split()) / max(1, len(en.split()))
    if r < 0.5 or r > 2.0:        # drop length-ratio outliers (likely bad translations)
        continue
    seen.add(sc); syn_pairs.append((sc, en))
print(f'kept {len(syn_pairs)} synthetic pairs (of {len(pool)})')
with open(f'{OUT}/synth_bt.tsv', 'w', encoding='utf-8') as f:
    for sc, en in syn_pairs:
        f.write(f'{sc}\t{en}\n')
print('saved synth_bt.tsv to', OUT)

## Fine-tune scn→en on real + synthetic

Warm-started from the merged bidirectional weights; a fresh LoRA learns the augmentation.

In [ ]:
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

torch.cuda.empty_cache()
EPOCHS = 1   # real 27k + synthetic; 1 pass keeps the synthetic noise in check

ft = get_peft_model(model, LoraConfig(r=32, lora_alpha=64, lora_dropout=0.05, bias='none',
                                      target_modules=['q_proj', 'k_proj', 'v_proj', 'out_proj'],
                                      task_type='SEQ_2_SEQ_LM'))
for p in ft.parameters():
    if p.requires_grad:
        p.data = p.data.float()
ft.config.use_cache = False
ft.enable_input_require_grads()
ft.print_trainable_parameters()

src = list(splits['train']['scn']) + [s for s, _ in syn_pairs]
tgt = list(splits['train']['en'])  + [e for _, e in syn_pairs]
tok.src_lang, tok.tgt_lang = 'scn_Latn', 'eng_Latn'
train_ds = Dataset.from_dict({'src': src, 'tgt': tgt}).map(
    lambda b: tok(b['src'], text_target=b['tgt'], max_length=128, truncation=True),
    batched=True, remove_columns=['src', 'tgt']).shuffle(seed=13)
print('training examples (real + synthetic):', len(train_ds))

args = Seq2SeqTrainingArguments(
    output_dir=f'{OUT}/trainer-bt-1.3B', num_train_epochs=EPOCHS,
    per_device_train_batch_size=4, gradient_accumulation_steps=4,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False},
    learning_rate=2e-4, fp16=True, logging_steps=100, save_strategy='no', report_to=[])
Seq2SeqTrainer(model=ft, args=args, train_dataset=train_ds,
               data_collator=DataCollatorForSeq2Seq(tok, model=ft)).train()

In [ ]:
model = ft.eval(); model.config.use_cache = True
report(translate(splits['test']['scn'], 'scn', 'en'), splits['test']['en'], 'BT scn->en 1.3B')
ft.save_pretrained(f'{OUT}/nllb-lora-bt-1.3B')
print('saved adapter + results_bt.json to', OUT)
print(json.dumps(results, indent=2, ensure_ascii=False))